# Projeto 3 - Iluminação 3D Avançada
## Disciplina de Computação Gráfica

#### Julia Cavallio Orlando - 14758721
#### Miguel Rodrigues Tomazini - 14599300

Este notebook integra o carregamento de malhas e texturas aos cálculos de iluminação baseados na equação de Phong. O projeto contempla múltiplas fontes de luz, isolamento fotométrico entre ambientes (paredes que bloqueiam luz) e controle em tempo real dos parâmetros através de shaders GLSL importados via módulos próprios.

### 1. Importação das Bibliotecas
Módulos necessários para manipulação matemática, leitura de imagens e interface com a API OpenGL.

In [101]:
import glfw
from OpenGL.GL import *
import numpy as np
import glm
import math
from PIL import Image

from shader_s import Shader

### 2. Inicialização do Contexto Gráfico
Configuração da janela da aplicação utilizando a biblioteca GLFW.

In [102]:
glfw.init()
glfw.window_hint(glfw.VISIBLE, glfw.FALSE)

altura = 1040
largura = 1280

window = glfw.create_window(largura, altura, "Projeto 3 - Iluminacao", None, None)

if (window == None):
    print("Falha ao criar a janela GLFW")
    glfw.terminate()
    
glfw.make_context_current(window)


(python:306539): Gtk-WARNING **: 21:41:51.541: gtk_disable_setlocale() must be called before gtk_init()


### 3. Compilação do Pipeline Programável (Shaders)
Carrega os arquivos externos GLSL, compila-os na GPU e cria o programa principal de renderização.

In [103]:
ourShader = Shader("vertex_shader.vs", "fragment_shader.fs")
ourShader.use()
program = ourShader.getProgram()

### 4. Interpretador de Malhas Wavefront (.obj) e Texturas
Estas funções leem as coordenadas espaciais, mapeamento UV e vetores normais necessários para a luz.

In [104]:
glEnable(GL_TEXTURE_2D)
glHint(GL_LINE_SMOOTH_HINT, GL_DONT_CARE)
glEnable(GL_BLEND)
glBlendFunc(GL_SRC_ALPHA, GL_ONE_MINUS_SRC_ALPHA)
glEnable(GL_LINE_SMOOTH)

global vertices_list, textures_coord_list, normals_list
vertices_list = []    
textures_coord_list = []
normals_list = []

def load_model_from_file(filename):
    """Extrai vértices (v), coordenadas de textura (vt), normais (vn) e faces (f) de um arquivo OBJ."""
    vertices, texture_coords, normals, faces = [], [], [], []
    material = None

    for line in open(filename, "r"):
        if line.startswith('#'): continue
        values = line.split()
        if not values: continue

        if values[0] == 'v':
            vertices.append(values[1:4])
        elif values[0] == 'vt':
            texture_coords.append(values[1:3])
        elif values[0] == 'vn':
            normals.append(values[1:4])
        elif values[0] in ('usemtl', 'usemat'):
            material = values[1]
        elif values[0] == 'f':
            face, face_texture, face_normal = [], [], []
            for v in values[1:]:
                w = v.split('/')
                face.append(int(w[0]))
                face_texture.append(int(w[1]) if len(w) >= 2 and len(w[1]) > 0 else 0)
                face_normal.append(int(w[2]) if len(w) >= 3 and len(w[2]) > 0 else 0)
            faces.append((face, face_texture, face_normal, material))

    return {'vertices': vertices, 'texture': texture_coords, 'normals': normals, 'faces': faces}

def load_texture_from_file(texture_id, img_textura):
    """Carrega a imagem via PIL e envia para a memória da textura no OpenGL."""
    glBindTexture(GL_TEXTURE_2D, texture_id)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_WRAP_S, GL_REPEAT)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_WRAP_T, GL_REPEAT)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MIN_FILTER, GL_LINEAR)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MAG_FILTER, GL_LINEAR)
    
    img = Image.open(img_textura)
    if img.mode in ('I;16', 'I;16B', 'I'):
        arr = np.array(img, dtype=np.uint16)
        arr = (arr >> 8).astype(np.uint8)
        img = Image.fromarray(arr, mode='L')
    img = img.convert("RGB")
    img_width, img_height = img.size
    image_data = img.tobytes("raw", "RGB", 0, -1)
    glTexImage2D(GL_TEXTURE_2D, 0, GL_RGB, img_width, img_height, 0, GL_RGB, GL_UNSIGNED_BYTE, image_data)

def circular_sliding_window_of_three(arr):
    """Converte polígonos de N vértices (ex: quadrados) em triângulos sequenciais."""
    if len(arr) == 3: return arr
    circular_arr = arr + [arr[0]]
    result = []
    for i in range(len(circular_arr) - 2):
        result.extend(circular_arr[i:i+3])
    return result

global numberTextures
numberTextures = 0

def load_obj_and_texture(objFile, texturesList, target_material=None):
    """Integra vértices, texturas e calcula normais fisicamente corretas via Produto Vetorial caso estejam ausentes no .obj."""
    modelo = load_model_from_file(objFile)
    verticeInicial = len(vertices_list)
    
    for face in modelo['faces']:
        if target_material is not None and face[3] != target_material:
            continue
            
        v_window = circular_sliding_window_of_three(face[0])
        t_window = circular_sliding_window_of_three(face[1])
        n_window = circular_sliding_window_of_three(face[2])
        
        # --- CÁLCULO DINÂMICO DE NORMAL ---
        # Se o OBJ não tiver normais (vn), usamos os 3 vértices do triângulo para descobrir para onde a face aponta
        if len(n_window) > 0 and (n_window[0] == 0 or len(modelo['normals']) == 0):
            v1 = np.array(modelo['vertices'][v_window[0] - 1], dtype=np.float32)
            v2 = np.array(modelo['vertices'][v_window[1] - 1], dtype=np.float32)
            v3 = np.array(modelo['vertices'][v_window[2] - 1], dtype=np.float32)
            
            # Produto vetorial das arestas
            normal_calculada = np.cross(v2 - v1, v3 - v1)
            norma_modulo = np.linalg.norm(normal_calculada)
            if norma_modulo > 0:
                normal_calculada = (normal_calculada / norma_modulo).tolist()
            else:
                normal_calculada = [0.0, 1.0, 0.0]
        else:
            normal_calculada = [0.0, 1.0, 0.0]
            
        # Adiciona os vértices filtrados aos Buffers da GPU
        for i in range(len(v_window)):
            vertices_list.append(modelo['vertices'][v_window[i] - 1])
            
            if t_window[i] > 0:
                u = float(modelo['texture'][t_window[i] - 1][0])
                v = float(modelo['texture'][t_window[i] - 1][1])
                textures_coord_list.append([u, v])
            else:
                textures_coord_list.append([0.0, 0.0])
                
            if n_window[i] > 0 and len(modelo['normals']) > 0:
                normals_list.append(modelo['normals'][n_window[i] - 1])
            else:
                normals_list.append(normal_calculada)
        
    verticeFinal = len(vertices_list)
    
    global numberTextures
    tex_start_id = numberTextures
    for tex in texturesList:
        load_texture_from_file(numberTextures, tex)
        numberTextures += 1
    
    return verticeInicial, verticeFinal - verticeInicial, tex_start_id

### 5. Gerenciamento de Entidades (Assets)
Cria um dicionário em memória que mapeia os objetos aos seus respectivos offsets dentro das listas globais da GPU.

In [105]:
game_assets = {}

def register_model(name, obj_path, tex_path, tex_scale_u=1.0, tex_scale_v=1.0, target_material=None):
    """Registra o modelo no dicionário com controle de repetição de textura (Tiling)."""
    if tex_scale_v == 1.0 and tex_scale_u != 1.0:
        tex_scale_v = tex_scale_u
        
    start, count, tex_id = load_obj_and_texture(obj_path, [tex_path], target_material)
    
    game_assets[name] = {
        'start': start, 
        'count': count, 
        'tex_id': tex_id, 
        'tex_scale_u': float(tex_scale_u), 
        'tex_scale_v': float(tex_scale_v)
    }

### 6. Instanciação do Cenário
Carregamento prático dos modelos e texturas, separando adequadamente o mundo interno do externo.

In [106]:
# --- AMBIENTE INTERNO ---
register_model('floor_int', 'objects/floor/floor.obj', 'objects/floor/floor_internal.jpg', tex_scale_u=10.0, tex_scale_v=10.0)
register_model('room_wall', 'objects/room/room.obj', 'objects/room/textures/wall.jpg',  target_material='wall', tex_scale_u=5.0)
register_model('room_wall2', 'objects/room/room.obj', 'objects/room/textures/wall2.jpg', target_material='wall2', tex_scale_u=5.0)
register_model('room_ceiling',    'objects/room/room.obj', 'objects/room/textures/floor.jpg', target_material='ceiling')
register_model('room_floor',    'objects/room/room.obj', 'objects/room/textures/wall.jpg',  target_material='floor', tex_scale_u=5.0)

register_model('bench', 'objects/bench/bench.obj', 'objects/bench/texture.tga')
register_model('treadmill', 'objects/treadmill/treadmill.obj', 'objects/treadmill/texture.png')
register_model('table_tennis', 'objects/tennis_table/tennis_table.obj', 'objects/tennis_table/texture.jpg')
register_model('ab_machine', 'objects/ab_machine/ab_machine.obj', 'objects/ab_machine/texture.png')
register_model('flashlight', 'objects/flashlight/flashlight.obj', 'objects/flashlight/texture.jpg')

# Ventilador (Sub-Meshes para permitir pintura emissiva individual)
register_model('body', 'objects/ceiling_fan/ceiling_fan.obj', 'objects/ceiling_fan/texture.png', target_material='body')
register_model('fan', 'objects/ceiling_fan/ceiling_fan.obj', 'objects/ceiling_fan/texture.png', target_material='fan')
register_model('lamp', 'objects/ceiling_fan/ceiling_fan.obj', 'objects/ceiling_fan/texture.png', target_material='lamp')

# --- AMBIENTE EXTERNO ---
register_model('skybox', 'objects/skybox/skybox.obj', 'objects/skybox/texture.png')
register_model('floor_ext', 'objects/floor/floor.obj', 'objects/floor/floor_external.jpg', tex_scale_u=20.0, tex_scale_v=20.0)
register_model('field', 'objects/field/field.obj', 'objects/field/texture.png')
register_model('ball', 'objects/ball/ball.obj', 'objects/ball/texture.png')
register_model('net', 'objects/net/net.obj', 'objects/net/texture.jpg')
register_model('footballer', 'objects/footballer/diniz.obj', 'objects/footballer/texture.png')
register_model('drone', 'objects/drone/drone.obj', 'objects/drone/texture.png')

# Esfera auxiliar reaproveitada como volume físico emissor de luz
register_model('light_sphere', 'objects/ball/ball.obj', 'objects/ball/texture.png')

# Arquibancada (Sub-Meshes com materiais distintos)
register_model('arquibancada_madeira', 'objects/bleacher/bleacher.obj', 'objects/bleacher/textures/wood.png', target_material='Wood')
register_model('arquibancada_metal',   'objects/bleacher/bleacher.obj', 'objects/bleacher/textures/metal.png', target_material='Metal')

/tmp/ipykernel_306539/3891500518.py:53: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = Image.fromarray(arr, mode='L')


### 7. Função Principal de Desenho
Comunica-se com o Shader para definir a matriz Model e os parâmetros fotométricos (Phong) do objeto atual antes da renderização geométrica.

In [107]:
def draw_model(name, angle=0.0, r_x=0.0, r_y=1.0, r_z=0.0, t_x=0.0, t_y=0.0, t_z=0.0, s_x=1.0, s_y=1.0, s_z=1.0, 
               escala=1.0, is_light=0, kd=0.8, ks=0.2, ns=32.0, obj_location=0, emit_color=(1.0, 1.0, 1.0)):
    asset = game_assets[name]
    
    # Transmite a matriz de Transformação 3D (Model)
    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x * escala, s_y * escala, s_z * escala)
    glUniformMatrix4fv(glGetUniformLocation(program, "model"), 1, GL_TRUE, mat_model)
    glUniform1f(glGetUniformLocation(program, "tex_scale_u"), asset['tex_scale_u'])
    glUniform1f(glGetUniformLocation(program, "tex_scale_v"), asset['tex_scale_v'])
    
    # Semântica de Isolamento de Luz (Externo vs Interno)
    glUniform1i(glGetUniformLocation(program, "obj_location"), obj_location)
    
    # Configurações do Modelo de Phong
    glUniform1i(glGetUniformLocation(program, "is_light_source"), is_light)
    glUniform3f(glGetUniformLocation(program, "emit_color"), *emit_color)
    glUniform1f(glGetUniformLocation(program, "kd"), kd)
    glUniform1f(glGetUniformLocation(program, "ks"), ks)
    glUniform1f(glGetUniformLocation(program, "ns"), ns)
    
    glBindTexture(GL_TEXTURE_2D, asset['tex_id'])
    glDrawArrays(GL_TRIANGLES, asset['start'], asset['count'])

### 8. Configuração dos Buffers da GPU (VBO e VAO)
Envia as listas Python processadas diretamente para a memória VRAM de alto desempenho.

In [108]:
buffer_VBO = glGenBuffers(3)

# 1. Buffer de Vértices Espaciais (x, y, z)
vertices = np.zeros(len(vertices_list), [("position", np.float32, 3)])
vertices['position'] = vertices_list
glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO[0])
glBufferData(GL_ARRAY_BUFFER, vertices.nbytes, vertices, GL_STATIC_DRAW)
stride = vertices.strides[0]
offset = ctypes.c_void_p(0)
loc_vertices = glGetAttribLocation(program, "position")
glEnableVertexAttribArray(loc_vertices)
glVertexAttribPointer(loc_vertices, 3, GL_FLOAT, False, stride, offset)

# 2. Buffer de Mapeamento UV (Texturas)
textures = np.zeros(len(textures_coord_list), [("position", np.float32, 2)])
textures['position'] = textures_coord_list
glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO[1])
glBufferData(GL_ARRAY_BUFFER, textures.nbytes, textures, GL_STATIC_DRAW)
stride = textures.strides[0]
offset = ctypes.c_void_p(0)
loc_texture_coord = glGetAttribLocation(program, "texture_coord")
glEnableVertexAttribArray(loc_texture_coord)
glVertexAttribPointer(loc_texture_coord, 2, GL_FLOAT, False, stride, offset)

# 3. Buffer de Vetores Normais (Iluminação de Phong)
normals = np.zeros(len(normals_list), [("position", np.float32, 3)])
normals['position'] = normals_list
glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO[2])
glBufferData(GL_ARRAY_BUFFER, normals.nbytes, normals, GL_STATIC_DRAW)
stride = normals.strides[0]
offset = ctypes.c_void_p(0)
loc_normals = glGetAttribLocation(program, "normal")
glEnableVertexAttribArray(loc_normals)
glVertexAttribPointer(loc_normals, 3, GL_FLOAT, False, stride, offset)

### 9. Entradas de Teclado, Câmera e Variáveis de Ambiente

In [109]:
# Configurações Iniciais da Câmera
cameraPos   = glm.vec3(0.0, 1.85, 0.0)
cameraFront = glm.vec3(0.0, 0.0, -1.0)
cameraUp    = glm.vec3(0.0, 1.0, 0.0)
firstMouse = True
yaw = -90.0 
pitch = 0.0
lastX = largura / 2
lastY = altura / 2
fov = 45.0

# Entidades Físicas do Jogo
ball_pos = glm.vec3(0.0, 0.0, -30.0)
ball_vel = glm.vec3(0.0, 0.0, 0.0)
ball_rotation = 0.0
ball_scale = 2.0

player_pos = glm.vec3(2.0, 0.0, -15.0)
drone_pos = glm.vec3(0.0, 5.0, -15.0)
game_over = False

# Variáveis de Controle de Tempo e Teclas
deltaTime = 0.0
lastFrame = 0.0
teclas = {}
polygonal_mode = False

# Constantes Dinâmicas e Físicas
vel_y = 0.0
is_jumping = False
gravity = -25.0
jump_force = 10.0
fan_rotation = 0.0
fan_speed = 150.0

# Variáveis de Gerenciamento Fotométrico
drone_light_on = 1
fan_light_on = 1
flash_light_on = 1
ambient_light_on = 1
ambient_intensity = 0.2
global_kd = 1.0
global_ks = 1.0

def key_event(window, key, scancode, action, mods):
    """Processa comandos de teclado únicos (gatilhos)."""
    global polygonal_mode, teclas
    global drone_light_on, fan_light_on, flash_light_on, ambient_light_on
    global ambient_intensity
    
    if action == glfw.PRESS:
        teclas[key] = True
        if key == glfw.KEY_ESCAPE: glfw.set_window_should_close(window, True)
        if key == glfw.KEY_P: polygonal_mode = not polygonal_mode
            
        # Acionamento Semântico Independente das Fontes de Luz
        if key == glfw.KEY_H: drone_light_on = 1 - drone_light_on
        if key == glfw.KEY_J: fan_light_on   = 1 - fan_light_on
        if key == glfw.KEY_K: flash_light_on = 1 - flash_light_on
        if key == glfw.KEY_L: ambient_light_on = 1 - ambient_light_on
            
        # Controle da Intensidade Global do Ambiente
        if key == glfw.KEY_EQUAL: ambient_intensity = min(1.0, ambient_intensity + 0.05)
        if key == glfw.KEY_MINUS: ambient_intensity = max(0.0, ambient_intensity - 0.05)
            
    elif action == glfw.RELEASE:
        teclas[key] = False

def mouse_callback(window, xpos, ypos):
    """Converte o deslocamento do mouse no vetor normalizado Look-At da câmera via ângulos de Euler."""
    global cameraFront, lastX, lastY, firstMouse, yaw, pitch
    if firstMouse:
        lastX = xpos
        lastY = ypos
        firstMouse = False
        
    xoffset = xpos - lastX
    yoffset = lastY - ypos
    lastX = xpos
    lastY = ypos

    sensitivity = 0.1
    yaw += xoffset * sensitivity
    pitch += yoffset * sensitivity
    
    # Limitação de Gimbal Lock
    if pitch > 89.0: pitch = 89.0
    if pitch < -89.0: pitch = -89.0
    
    front = glm.vec3()
    front.x = glm.cos(glm.radians(yaw)) * glm.cos(glm.radians(pitch))
    front.y = glm.sin(glm.radians(pitch))
    front.z = glm.sin(glm.radians(yaw)) * glm.cos(glm.radians(pitch))
    cameraFront = glm.normalize(front)

glfw.set_key_callback(window, key_event)
glfw.set_cursor_pos_callback(window, mouse_callback)
glfw.set_input_mode(window, glfw.CURSOR, glfw.CURSOR_DISABLED)

### 10. Matrizes de Visualização (MVP) e Colisão

In [110]:
def model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z):
    """Gera a matriz de translação, rotação e escala da geometria no espaço do mundo."""
    angle = math.radians(angle)
    matrix_transform = glm.mat4(1.0)
    matrix_transform = glm.translate(matrix_transform, glm.vec3(t_x, t_y, t_z))    
    if angle != 0:
        matrix_transform = glm.rotate(matrix_transform, angle, glm.vec3(r_x, r_y, r_z))
    matrix_transform = glm.scale(matrix_transform, glm.vec3(s_x, s_y, s_z))
    return np.array(matrix_transform)

def view():
    """Gera a matriz da câmera baseada na posição do observador."""
    global cameraPos, cameraFront, cameraUp
    return np.array(glm.lookAt(cameraPos, cameraPos + cameraFront, cameraUp))

def projection():
    """Gera a matriz de perspectiva matemática."""
    global altura, largura
    return np.array(glm.perspective(glm.radians(fov), largura/altura, 0.1, 1000.0))

def check_collision(x, z, radius=1.0):
    """Algoritmo AABB para impedir a travessia de barreiras sólidas."""
    if x < -30.0 or x > 30.0 or z < -60.0 or z > 60.0:
        return True
    if 0.0 - radius < z < 0.75 + radius:
        if x < -6.0 + radius or x > 6.0 - radius:
            return True 
    if z >= 0.75 - radius and z <= 30.0:
        if z > 29.25 - radius: return True
        if x < -28.5 + radius: return True
        if x > 28.5 - radius:  return True
    return False

### 11. Loop Principal (Pipeline e Renderização)
Mantém a estabilidade do tempo e executa consecutivamente a Física, a configuração dos Buffers Iluminados e os Draw Calls.

In [111]:
glfw.show_window(window)
glEnable(GL_DEPTH_TEST)

lastFrame = glfw.get_time()
rot_axis_x, rot_axis_z = 0.0, 1.0 

ball_vel_y = 0.0
player_angle = 180.0

while not glfw.window_should_close(window):
    currentFrame = glfw.get_time()
    deltaTime = currentFrame - lastFrame
    lastFrame = currentFrame
    glfw.poll_events()
    
    # ------------------------------------------------------------------
    # 1. CONTROLES DE ILUMINAÇÃO (REFLEXÃO DIFUSA E ESPECULAR)
    # ------------------------------------------------------------------
    # Atualiza coeficientes globais via teclado com uso responsivo via DeltaTime
    if teclas.get(glfw.KEY_I, False): global_kd = min(3.0, global_kd + 0.5 * deltaTime)
    if teclas.get(glfw.KEY_U, False): global_kd = max(0.0, global_kd - 0.5 * deltaTime)
        
    if teclas.get(glfw.KEY_T, False): global_ks = min(3.0, global_ks + 0.5 * deltaTime)
    if teclas.get(glfw.KEY_Y, False): global_ks = max(0.0, global_ks - 0.5 * deltaTime)

    # ------------------------------------------------------------------
    # 2. FÍSICA CINEMÁTICA E COLISÃO DA CÂMERA
    # ------------------------------------------------------------------
    if glfw.get_key(window, glfw.KEY_SPACE) == glfw.PRESS and not is_jumping:
        vel_y = jump_force
        is_jumping = True
    vel_y += gravity * deltaTime
    cameraPos.y += vel_y * deltaTime
    if cameraPos.y <= 1.85:
        cameraPos.y = 1.85
        vel_y = 0.0
        is_jumping = False

    # Translação Contínua (WASD) bloqueada via AABB para evitar travessias
    cameraSpeed = 10.0 * deltaTime
    new_pos = glm.vec3(cameraPos)
    front_xz = glm.normalize(glm.vec3(cameraFront.x, 0.0, cameraFront.z))
    if teclas.get(glfw.KEY_W, False): new_pos += cameraSpeed * front_xz
    if teclas.get(glfw.KEY_S, False): new_pos -= cameraSpeed * front_xz
    if teclas.get(glfw.KEY_A, False): new_pos -= glm.normalize(glm.cross(front_xz, cameraUp)) * cameraSpeed
    if teclas.get(glfw.KEY_D, False): new_pos += glm.normalize(glm.cross(front_xz, cameraUp)) * cameraSpeed
    if not check_collision(new_pos.x, cameraPos.z): cameraPos.x = new_pos.x
    if not check_collision(cameraPos.x, new_pos.z): cameraPos.z = new_pos.z

    # ------------------------------------------------------------------
    # 3. FÍSICA E INTERAÇÃO COM A BOLA
    # ------------------------------------------------------------------
    raio_bola = ball_scale / 2.0
    
    # Identifica colisão in-game entre o vetor observador e o raio físico da bola
    if glm.length(glm.vec3(cameraPos.x, 0.0, cameraPos.z) - glm.vec3(ball_pos.x, 0.0, ball_pos.z)) < raio_bola + 1.0:
        ball_vel = glm.normalize(glm.vec3(ball_pos.x, 0.0, ball_pos.z) - glm.vec3(cameraPos.x, 0.0, cameraPos.z)) * 15.0
        ball_vel_y = 8.0
        
    speed = glm.length(ball_vel)
    if speed > 0.01:
        new_ball_x = ball_pos.x + ball_vel.x * deltaTime
        new_ball_z = ball_pos.z + ball_vel.z * deltaTime
        bounce = 0.8 
        
        if check_collision(new_ball_x, ball_pos.z, radius=raio_bola): ball_vel.x *= -bounce
        else: ball_pos.x = new_ball_x
            
        if check_collision(ball_pos.x, new_ball_z, radius=raio_bola): ball_vel.z *= -bounce
        else: ball_pos.z = new_ball_z
            
        ball_vel -= ball_vel * 2.0 * deltaTime
        ball_rotation += (speed * deltaTime / raio_bola) * (180.0 / math.pi)
        rot_axis_x = ball_vel.z / speed
        rot_axis_z = -ball_vel.x / speed
        
    # Vetor Gravitacional na malha da bola
    ball_vel_y += gravity * deltaTime
    ball_pos.y += ball_vel_y * deltaTime
    limite_chao = (ball_scale / 10)
    if ball_pos.y <= limite_chao:
        ball_pos.y = limite_chao
        if ball_vel_y < -2.0:
            ball_vel_y *= -0.7
        else:
            ball_vel_y = 0.0

    # ------------------------------------------------------------------
    # 4. COMPORTAMENTOS DE IA (JOGADOR E DRONE)
    # ------------------------------------------------------------------
    if not game_over:
        dist_vec_ball = ball_pos - player_pos
        dist_vec_ball.y = 0 
        dist_to_ball = glm.length(dist_vec_ball)
        
        # Transformação esférica do Look-At para a IA observar organicamente a bola
        player_angle = math.degrees(math.atan2(dist_vec_ball.x, dist_vec_ball.z))
        
        if dist_to_ball > 0.5:
            player_pos += glm.normalize(dist_vec_ball) * 6.0 * deltaTime
        else:
            ball_vel = glm.normalize(dist_vec_ball) * 30.0
            ball_vel_y = 5.0

    # Interpolação Linear de perseguição do Drone (Comportamento Exterior)
    if cameraPos.z < 0.0:  
        vetor_distancia = glm.vec3(cameraPos.x, cameraPos.y + 4.0, cameraPos.z) - drone_pos
        if glm.length(vetor_distancia) > 0.05:
            drone_pos += glm.normalize(vetor_distancia) * 5.0 * deltaTime
    else:
        vetor_retorno = glm.vec3(0.0, 5.0, -2.0) - drone_pos
        if glm.length(vetor_retorno) > 0.05:
            drone_pos += glm.normalize(vetor_retorno) * 4.0 * deltaTime

    # ------------------------------------------------------------------
    # 5. GERENCIAMENTO DINÂMICO DE OBJETOS
    # ------------------------------------------------------------------
    if teclas.get(glfw.KEY_E, False): ball_scale += 0.5 * deltaTime
    if teclas.get(glfw.KEY_Q, False): ball_scale = max(0.1, ball_scale - 0.5 * deltaTime)
    if glfw.get_key(window, glfw.KEY_RIGHT) == glfw.PRESS: fan_speed += 100.0 * deltaTime
    if glfw.get_key(window, glfw.KEY_LEFT) == glfw.PRESS: fan_speed -= 100.0 * deltaTime
        
    glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT)
    glClearColor(0.2, 0.2, 0.2, 1.0)
    glPolygonMode(GL_FRONT_AND_BACK, GL_LINE if polygonal_mode else GL_FILL)
    fan_rotation += fan_speed * deltaTime

    # ------------------------------------------------------------------
    # 6. ENVIO DE VETORES DE LUZ (PHONG UNIFORMS)
    # ------------------------------------------------------------------
    glUniform1i(glGetUniformLocation(program, "ambientActive"), ambient_light_on)
    glUniform1f(glGetUniformLocation(program, "ambientIntensity"), ambient_intensity)
    glUniform1f(glGetUniformLocation(program, "global_kd"), global_kd)
    glUniform1f(glGetUniformLocation(program, "global_ks"), global_ks)
    glUniform3f(glGetUniformLocation(program, "viewPos"), cameraPos.x, cameraPos.y, cameraPos.z)

    # Configurações Fonte Lúminosa (Móvel Externa)
    luz_drone_pos = glm.vec3(drone_pos.x, drone_pos.y - 0.08, drone_pos.z)
    glUniform3f(glGetUniformLocation(program, "lights[0].position"), *luz_drone_pos)
    glUniform3f(glGetUniformLocation(program, "lights[0].color"), 2.0, 2.0, 2.0)
    glUniform1i(glGetUniformLocation(program, "lights[0].type"), 0) 
    glUniform1i(glGetUniformLocation(program, "lights[0].is_on"), drone_light_on)
    glUniform1f(glGetUniformLocation(program, "lights[0].cutOff"), -2.0)

    # Configurações Fontes de Luz Internas (Esféricas Livres)
    for i in range(3):
        x_fan = (i*15) - 15
        luz_fan_pos = glm.vec3(x_fan, 4.4, 15.0) 
        glUniform3f(glGetUniformLocation(program, f"lights[{i+1}].position"), *luz_fan_pos)
        glUniform3f(glGetUniformLocation(program, f"lights[{i+1}].color"), 1.2, 1.2, 1.0)
        glUniform1i(glGetUniformLocation(program, f"lights[{i+1}].type"), 1)
        glUniform1i(glGetUniformLocation(program, f"lights[{i+1}].is_on"), fan_light_on)
        glUniform1f(glGetUniformLocation(program, f"lights[{i+1}].cutOff"), -2.0)
    
    pos_lanterna = glm.vec3(19.0, 0.89, 15.0)
    pos_luz_lanterna = glm.vec3(18.78, 0.89, 15.2) 
    dir_lanterna = glm.vec3(-1.0, -0.3, 0.5) 
    
    glUniform3f(glGetUniformLocation(program, "lights[4].position"), *pos_luz_lanterna)
    glUniform3f(glGetUniformLocation(program, "lights[4].color"), 4.0, 0.2, 0.2)
    glUniform1i(glGetUniformLocation(program, "lights[4].type"), 1)
    glUniform1i(glGetUniformLocation(program, "lights[4].is_on"), flash_light_on)
    glUniform3f(glGetUniformLocation(program, "lights[4].direction"), *dir_lanterna)
    glUniform1f(glGetUniformLocation(program, "lights[4].cutOff"), math.cos(math.radians(25.0)))
    glUniform1f(glGetUniformLocation(program, "lights[4].outerCutOff"), math.cos(math.radians(35.0)))

    # ------------------------------------------------------------------
    # 7. DRAW CALLS DA GEOMETRIA
    # ------------------------------------------------------------------
    
    # Desenho Visual dos Pontos Emissores de Fótons (Malha da Luz)
    if drone_light_on:
        draw_model('light_sphere', t_x=luz_drone_pos.x, t_y=luz_drone_pos.y, t_z=luz_drone_pos.z, 
                   escala=0.08, is_light=1, emit_color=(1.0, 1.0, 1.0), obj_location=0)
        
    if flash_light_on:
        draw_model('light_sphere', t_x=pos_luz_lanterna.x, t_y=pos_luz_lanterna.y, t_z=pos_luz_lanterna.z, 
                   escala=0.2, is_light=1, emit_color=(2.0, 0.2, 0.2), obj_location=1)

    # Geometria Delimitadora
    draw_model('room_wall', t_z=15, t_y=0, s_x=30, s_y=5, s_z=15, ks=0.1, ns=8.0, obj_location=2)
    draw_model('room_wall2', t_z=15, t_y=0, s_x=30, s_y=5, s_z=15, ks=0.1, ns=8.0, obj_location=2)
    draw_model('room_ceiling',    t_z=15, t_y=0, s_x=30, s_y=5, s_z=15, ks=0.1, ns=8.0, obj_location=2)
    draw_model('room_floor',    t_z=15, t_y=0, s_x=30, s_y=5, s_z=15, ks=0.3, ns=16.0, obj_location=2)

    # Modelos de Interiores
    draw_model('floor_int', t_z=15, t_y=0.01, s_x=60, s_z=30, escala=0.5, ks=0.1, obj_location=1)
    draw_model('bench', t_x=-10.0, t_y=0.0, t_z=15.0, escala=0.015, ks=0.2, obj_location=1)
    draw_model('bench', t_x=10.0, t_y=0.0, t_z=15.0, escala=0.015, ks=0.2, obj_location=1)
    draw_model('bench', t_x=20.0, t_y=0.0, t_z=15.0, escala=0.015, ks=0.2, obj_location=1)
    draw_model('bench', t_x=-20.0, t_y=0.0, t_z=15.0, escala=0.015, ks=0.2, obj_location=1)
    draw_model('flashlight', t_x=pos_lanterna.x, t_y=pos_lanterna.y, t_z=pos_lanterna.z, 
               escala=0.04, angle=115.0, r_y=0.7, r_z=1.0, ks=0.8, ns=64.0, obj_location=1)
    for i in range(11):
        draw_model('treadmill', t_x=-25.0 + i*5.0, t_y=0.0, t_z=27.0, escala=0.03, ks=0.6, obj_location=1)
    draw_model('table_tennis', t_x=0.0, t_y=0.8, t_z=15.0, escala=0.015, ks=0.5, obj_location=1)
    for i in range(5):
        draw_model('ab_machine', t_x=-25.0 + i * 4.0, t_y=0.0, t_z=5.0, angle=-180.0, r_y=1.0, escala=0.024, ks=0.6, obj_location=1)
        draw_model('ab_machine', t_x=25.0 - i * 4.0, t_y=0.0, t_z=5.0, angle=-180.0, r_y=1.0, escala=0.024, ks=0.6, obj_location=1)
    for i in range(3):
        x_fan = (i*15)-15
        draw_model('body', t_x=x_fan, t_y=2.5, t_z=15, escala=0.5, ks=0.8, obj_location=1)
        draw_model('fan', t_x=x_fan, t_y=2.5, t_z=15, angle=fan_rotation, r_y=1, escala=0.5, ks=0.8, obj_location=1)
        if fan_light_on:
            draw_model('lamp', t_x=x_fan, t_y=2.15, t_z=15, angle=fan_rotation, r_y=1, escala=0.6, is_light=1, obj_location=1, emit_color=(1.2, 1.2, 1.0))

    # Modelos Exteriores
    draw_model('skybox', t_x=cameraPos.x, t_y=cameraPos.y, t_z=cameraPos.z, escala=500.0, ks=0.0, obj_location=0)
    draw_model('field', t_y=0.01, t_z=-30, s_x=60, s_z=60, escala=0.5, ks=0.1, ns=4.0, obj_location=0)
    draw_model('floor_ext', t_z=-50, s_x=160, s_z=200, escala=0.5, ks=0.1, obj_location=0)
    draw_model('net', t_x=29.4, t_z=-30, angle=180.0, r_y=1.0, ks=0.9, ns=64.0, obj_location=0)
    draw_model('net', t_x=-29.4, t_z=-30, ks=0.9, ns=64.0, obj_location=0)
    draw_model('drone', t_x=drone_pos.x, t_y=drone_pos.y, t_z=drone_pos.z, angle=180.0, r_y=1.0, escala=0.1, ks=0.2, ns=64.0, obj_location=0)
    draw_model('footballer', t_x=player_pos.x, t_y=player_pos.y, t_z=player_pos.z, angle=player_angle, r_y=1.0, escala=0.3, ks=0.3, obj_location=0)
    draw_model('ball', t_x=ball_pos.x, t_y=ball_pos.y, t_z=ball_pos.z, escala=ball_scale, angle=ball_rotation, r_x=rot_axis_x, r_y=0.0, r_z=rot_axis_z, ks=0.6, ns=32.0, obj_location=0)

    for i in range(6):
        draw_model('arquibancada_madeira', t_x=35, t_z=(-10*i)-5, s_y=0.75, escala=2.5, angle=-90.0, r_y=1.0, ks=0.2, obj_location=0)
        draw_model('arquibancada_metal',   t_x=35, t_z=(-10*i)-5, s_y=0.75, escala=2.5, angle=-90.0, r_y=1.0, ks=0.8, obj_location=0)
        draw_model('arquibancada_madeira', t_x=-35, t_z=(-10*i)-5, s_y=0.75, escala=2.5, angle=90.0, r_y=1.0, ks=0.2, obj_location=0)
        draw_model('arquibancada_metal',   t_x=-35, t_z=(-10*i)-5, s_y=0.75, escala=2.5, angle=90.0, r_y=1.0, ks=0.8, obj_location=0)

    for i in range(7):
        draw_model('arquibancada_madeira', t_x=(10*i)-30, t_z=-65, s_y=0.75, escala=2.5, ks=0.2, obj_location=0)
        draw_model('arquibancada_metal',   t_x=(10*i)-30, t_z=-65, s_y=0.75, escala=2.5, ks=0.8, obj_location=0)

    # Envio das Matrizes Finais e Renderização do Frame
    glUniformMatrix4fv(glGetUniformLocation(program, "view"), 1, GL_TRUE, view())
    glUniformMatrix4fv(glGetUniformLocation(program, "projection"), 1, GL_TRUE, projection())    
    glfw.swap_buffers(window)

glfw.terminate()